In [4]:
import gymnasium as gym
import tensorflow as tf
import numpy as np
from collections import deque

In [5]:
env=gym.make("CartPole-v1")
input_shape=[4]
n_outputs=2

In [6]:
model=tf.keras.models.Sequential([
    tf.keras.layers.Dense(32,activation="elu",input_shape=input_shape),
    tf.keras.layers.Dense(32,activation="elu"),
    tf.keras.layers.Dense(n_outputs)
])

c:\Users\moon\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
def epsilon_greedy_policy (state,epsilon=0):
    if np.random.rand() <=epsilon:
        return np.random.randint(2)
    else:
        q_values=model.predict(state[np.newaxis])
        return np.argmax(q_values[0])


In [8]:
replay_buffer=deque(maxlen=2000)

In [9]:
def sample_experiences (batch_size):
    indices=np.random.randint(len(replay_buffer),size=batch_size)
    batch=[replay_buffer[index] for index in indices]
    states,actions,rewards,next_state,dones=[np.array([experience[field_index] for experience in batch]) for field_index in range(5)]
    return states,actions,rewards,next_state,dones

In [10]:
def play_one_step (env,state,epsilon):
    action=epsilon_greedy_policy(state,epsilon)
    next_state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    replay_buffer.append((state,action,reward,next_state,done))
    return next_state,reward,done,info
    

In [11]:
batch_size=32
discount_factor=0.95
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
loss_fn = tf.keras.losses.MeanSquaredError()

In [12]:
def training_step (batch_size):
    experiences=sample_experiences(batch_size)
    states,actions,rewards,next_states,dones=experiences
    next_q_values=model.predict(next_states)
    max_next_q_values=np.max(next_q_values,axis=1)
    target_q_values=(rewards+(1-dones)*discount_factor*max_next_q_values)
    target_q_values=target_q_values.reshape(1,-1)
    mask=tf.one_hot(actions,n_outputs)
    with tf.GradientTape() as tape:
        all_q_values=model(states)
        q_values=tf.reduce_sum(all_q_values* mask,axis=1,keepdims=True)
        loss=tf.reduce_mean(loss_fn(target_q_values,q_values))
        grads=tape.gradient(loss,model.trainable_variables)
        optimizer.apply_gradients(zip(grads,model.trainable_variables))


In [ ]:
for eposide in range(600):
    obs, info = env.reset()
    for step in range(200):
        epsilon = max(1 - eposide / 500, 0.01)
        obs, reward, done, info = play_one_step(env, obs, epsilon)
        if done:
            break
    if eposide >50:
        training_step(batch_size)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━